# 📓 Cas d'usage IA — [TITRE DU PROJET]

**Auteur·e** : `Romain Busuttil`  
**Promo** : ATOS Atlas IA — Parcours 2 (Pros IT)  
**Date début** : `16/07/2026`  
**Dernière mise à jour** : `16/07/2026`

---

## 0. Imports & configuration

In [3]:
# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Affichage
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

# Imports ML (à activer au fur et à mesure)
# from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline

## 0.5 Traçabilité de la session

In [4]:
import sys
import subprocess
from datetime import datetime

# --- Métadonnées du dataset ---
DATASET_NAME = "dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv"
DATASET_SOURCE = "../data"
DATASET_VERSION = "2026-07-08"

# --- Métadonnées de la session ---
session_date = datetime.now().isoformat(timespec="minutes")
py_version = sys.version.split()[0]

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    git_commit = "non disponible (notebook hors repo Git)"

print(f"Date session : {session_date}")
print(f"Python       : {py_version}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Git commit   : {git_commit}")
print(f"Dataset      : {DATASET_NAME} (source={DATASET_SOURCE}, version={DATASET_VERSION})")

Date session : 2026-07-20T16:10
Python       : 3.11.9
NumPy        : 1.26.4
Pandas       : 2.2.2
Git commit   : 38f8bfd
Dataset      : dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv (source=../data, version=2026-07-08)


---
## 1. Cadrage métier & problème

### 1.1 Contexte client

Le client est une agence nationale de l’emploi (ex : France Travail), donc c'est un service public chargé de la politique de l’emploi. Elle accompagne les demandeurs d’emploi dans leur retour à l’activité en mobilisant des dispositifs adaptés (formations, suivi renforcé, aides). Son secteur est régulé, avec des obligations de neutralité, d’égalité de traitement et de transparence, encadrées par le RGPD et l’AI Act.
Son problème métier principal est d’orienter rapidement et efficacement chaque usager dès le premier entretien, malgré un volume élevé de dossiers et une forte hétérogénéité des profils. Une mauvaise orientation, notamment pour les personnes à risque de chômage longue durée, peut entraîner uncoût humain et financier élevé.
Le recours à l’intelligence artificielle est motivé par la volonté d’identifier rapidement les situations à risque, et d’optimiser l’allocation des ressources d’accompagnement, et donc assister les conseillers dans leur prise de décision.

### 1.2 Énoncé du problème IA

Prédire le délai de retour à l’emploi (rapide <6 mois, moyen 6-12 mois, long >12 mois) à partir de données multimodales comprenant des variables tabulaires (âge, diplôme, expérience, statut, géographie) et du texte libre issu de la synthèse d’entretien.
Prédire la classe de retour à l’emploi (0, 1 ou 2) à partir des caractéristiques individuelles et du contenu textuel décrivant la situation de l’usager.

### 1.3 Bénéficiaires & impacts

Direct : Les conseillers de l’agence de l’emploi, qui s’appuient sur les prédictions pour orienter les demandeurs d’emploi, prioriser les suivis et décider des dispositifs d’accompagnement.
Indirect : Les demandeurs d’emploi, qui subissent directement les conséquences des prédictions (type d’accompagnement, intensité du suivi, accès à certaines aides).

### 1.4 Critères de succès

| Type | Critère | Cible initiale (client) | Cible révisée après EDA | Justification de la révision |
|---|---|---|---|---|
| Métier | Réduire le délai moyen de retour à l’emploi pour les profils à risque | -20% |  |  |
| Modèle | Accuracy, F1-score macro et Matrice de confusion | Accuracy ≥ 80%, F1-score macro ≥ 0.75, Recall ≥ 85% pour classe 2 |  | … |
| Opérationnel | Temps de réponse API en production | < 250 ms |  | … |

Les cibles initiales n'étant pas définies dans le sujet, elles sont définies à partir d’objectifs métier ambitieux. Elles seront ensuite potentiellement révisées après l'analyse exploratoire des données.

### 1.5 Risques éthiques & réglementaires anticipés

**🔐 Variables sensibles identifiées** :
- `nationalite_hors_ue` : données sensible pouvant entrainer une risque de discrimination.
- `code_insee_commune` : peut donner des informations sur l'origine sociale, le niveau de vie. C'est une variable proxy.
- `age` : critère de discrimination (séniors / jeunes)
- `est_allocataire` : critère de discrimination (précarité)
- `synthese_entretien` : texte libre qui pourrait contenir des informations personnelle (situation familiale, état de santé, ...)

**Biais potentiels** :
- Les synthèses des conseillers peuvent déjà contenir des biais (anticipation des difficultés a retrouver un travail).
- Les données proviennent d'une agence spécifique et donc non représentatives du territoire national.
- Corrélation entre `nationalite_hors_ue` et `code_insee_commune` qui donnerait des informations sur les quartiers défavorisés.
- Certaines populations sous-représentées dans les 2500 observations (exemple `nationalite_hors_ue`, `age`, `niveau_diplome`).
- Classe déséquilibré : une catégorie pas prise en compte par le modèle.

**RGPD** :
- Pseudonymisation : remplacer `usager_id` par un hash (ex : SHA-256) pour éviter la ré-identification.
- Droit à l'oubli : permettre aux usagers de supprimer leurs données après 3 ans.
- Minimisation des données : ne garder que les variables nécessaires
- Analyse d'impact : obligatoire pour les traitements à haut risque.

**AI Act** :
Cas d’usage potentiellement à haut risque car lié à l’emploi et à l’accès à des services publics.
Nécessité de supervision humaine, documentation, robustesse, gestion des logs et suivi des dérives.

**Droit sectoriel** :
Attention à la discrimination (`nationalite_hors_ue`, `age`, `code_insee_commune`)

### 1.6 📝 Synthèse cadrage

Le projet vise à aider une agence publique de l’emploi à mieux orienter les demandeurs d’emploi dès le premier entretien.  
La tâche IA retenue est une classification en 3 classes (<6 mois*, *6-12 mois*, *>12 mois) du délai de retour à l’emploi à partir de données tabulaires et de texte libre.  
Le critère de succès dominant est la qualité de détection des profils à risque, avec un bon F1-score macro et une matrice de confusion équilibrée.  
Le principal risque éthique identifié est la discrimination indirecte via des variables sensibles ou proxy, dans un contexte de traitement de données personnelles.

---
## 2. Identification & acquisition des données

### 2.1 Sources

| Source | Type | Volumétrie | Format | Accès | Date extraction |
|---|---|---|---|---|---|
| Fichier fourni examen :  dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143)local | 2 500 lignes × 10 colonnes | `.csv` | Lecture seule | `08/07/2026` |


In [7]:
# 2.2 Chargement
DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv")
df.shape, df.columns.tolist()

((2500, 10),
 ['usager_id',
  'age',
  'niveau_diplome',
  'anciennete_poste_ans',
  'code_rome_vise',
  'code_insee_commune',
  'est_allocataire',
  'nationalite_hors_ue',
  'synthese_entretien',
  'classe_retour_emploi'])

### 2.3 Dictionnaire de variables

| Variable | Description | Type | Plage / valeurs | Cible ? | Sensible ? |
|---|---|---|---|---|---|
| `usager_id` | IdenIdentifiant unique du demandeur d'emploi | catégoriel (ID) | `ID_0000` … `ID_2499` | ❌ | ✅ (pseudonyme à hasher si utilisé) |
| `age` | Âge de l'usager en années | numérique (entier) | 18 – 63 ans | ❌ | ✅ (discrimination âge) |
| `niveau_diplome` | Plus haut niveau de diplôme obtenu (Donnée manquante potentielle) | catégoriel (ordinal) | `Sans diplôme`, `Bac`, `Bac+2`, `Bac+5` | ❌ | ❌ |
| `anciennete_poste_ans` | Nombre d'années d'expérience dans le dernier emploi occupé | numérique (continue) | 0,0 – 22,6 ans | ❌ | ❌ |
| `code_rome_vise` | Code à 5 caractères du Répertoire Opérationnel des Métiers et des Emplois ciblé par l'usager | catégoriel (nominal) | codes ROME (ex : `D1503`, `G1302`, …) | ❌ | ❌ |
| `code_insee_commune` | Code officiel géographique INSEE de la commune de résidence | catégoriel (nominal) | codes INSEE communes françaises | ❌ | ✅ (variable proxy) |
| `est_allocataire` | Statut d'indemnisation de l'usager | binaire | 0 = non, 1 = oui | ❌ | ✅ (indicateur de précarité) |
| `nationalite_hors_ue` | Nationalité hors Union Européenne | binaire | 0 = UE, 1 = hors UE | ❌ | ✅ (donnée sensible RGPD) |
| `synthese_entretien` | Texte libre contenant les notes textuelles du conseiller lors de l'entretien  | texte libre | variable (phrases courtes) | ❌ | ✅ (peut contenir données personnelles) |
| `classe_retour_emploi` | **Délai de retour à l'emploi (variable cible)** | catégoriel (ordinal) | 0 = Rapide <6 mois, 1 = Moyen 6–12 mois, 2 = Long >12 mois | ✅ | ❌ |

### 2.4 📝 Synthèse acquisition

Le jeu de données provient d'un fichier CSV fourni dans le cadre de l'examen CISIA. Il contient **2 500 observations** et **10 colonnes** : 9 features (tabulaires + texte libre) et 1 variable cible de classification à 3 classes.  
Plusieurs variables sensibles ont été identifiées dès cette étape (`nationalite_hors_ue`, `age`, `code_insee_commune`, `synthese_entretien`, `est_allocataire`), nécessitant une attention particulière aux biais et au respect du RGPD.


---
## 3. Exploration & analyse des données (EDA)